In [ ]:
# OTHELLO bootstrap: make the package importable from notebooks/
import sys
from pathlib import Path
_repo_root = Path.cwd().parent.resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
import pandas as pd


In [23]:
names = ("qald", "mintaka", "hotpot")
paths = [f"datasets/{name}.json" for name in names]

In [24]:

def processed_df(path):
  if path.split('.')[-1] =='json':
    data = pd.read_json(path)
  else:
    data = pd.read_csv(path)


  unwanted ={'mintaka':['id', 'translations', 'questionEntity', 'category', 'complexityType', 'answer'], 'qald': None,'hotpot':['supporting_facts', 'level', 'context', 'answer', '_id','type']}
  #Create a dictionary that stores the name of the dataset as the key, and the list of unwanted columns as a key, and then call data.drop(columns = unwanted['dataset_name'])

  pre =path.split('/')
  name = pre[-1].split('.')[0]


  if name in unwanted.keys():
    cols = unwanted[name]


  if name =='qald':
    questions = []
    for index, row in data.iterrows():
      question = row['questions']['question'][0]['string']
      questions.append(question)
    data = pd.DataFrame({'question':questions})


  if cols !=None:
    data.drop(columns = cols,inplace=True)

  to_translate = data[:100]

  return to_translate, name

In [25]:
frames  = {}

for path in paths:
  df, name = processed_df(path)
  frames[name] = df

In [26]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

In [27]:
## translation func
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(api_key=OPENAI_API_KEY)

# Few-shot examples for translation to AAVE
few_shot_examples = [
    "I was bewildered, but I knew dat it was no gud asking his ass to explain.",
    "Cochran pontificated windily for da camera.",
    "I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go."
]

#Translation API Call + Prompt

def translate_to_aave(sae_text, few_shot_examples):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that translates Standard American English (SAE) to African American Vernacular English (AAVE)."},
        {"role": "user", "content": (
            "Translate the following sentence from Standard English to African American Vernacular English (AAVE). "
            "Ensure the translation maintains the structure of the original sentence without adding extra information.\n\n"
            "Examples for reference:\n"
            "1. I was bewildered, but I knew dat it was no gud asking his ass to explain.\n"
            "2. Cochran pontificated windily for da camera.\n"
            "3. I don’t want them to follow in my footsteps, as I ain’t go to no college, but I want them to go.\n\n"

            f"Text: {sae_text}\n\n"
            "AAVE Translation:"
        )}
    ]

# Model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=150,
        temperature=0.7
    )

    return response.choices[0].message.content.strip()



# What columns are being made in what order
translated_data = pd.DataFrame(columns=['AAVE Question'])


In [28]:
processed = {}

# todo: if `translations/` already exists, delete and remake

for name, to_translate in frames.items():
  translated_data = pd.DataFrame(columns=['AAVE Question'])
  for index, row in to_translate.iterrows(): #Iterating over untranslated dataframe
    sae_question = row['question']
    aave_question = translate_to_aave(sae_question, few_shot_examples)
    print(f"Processed row {index + 1} question")

    # Creating new dataframe from results for csv file

    translated_data = pd.concat([translated_data, pd.DataFrame({
        'AAVE Question': [aave_question]
    })], ignore_index=True)

  processed[name] = translated_data
  translated_file_path = f'translations/aave_{name}.csv'
  translated_data.to_csv(translated_file_path, index=False)

  print(f"Translations completed and saved to {translated_file_path}")
  







Processed row 1 question
Processed row 2 question
Processed row 3 question
Processed row 4 question
Processed row 5 question
Processed row 6 question
Processed row 7 question
Processed row 8 question
Processed row 9 question
Processed row 10 question
Processed row 11 question
Processed row 12 question
Processed row 13 question
Processed row 14 question
Processed row 15 question
Processed row 16 question
Processed row 17 question
Processed row 18 question
Processed row 19 question
Processed row 20 question
Processed row 21 question
Processed row 22 question
Processed row 23 question
Processed row 24 question
Processed row 25 question
Processed row 26 question
Processed row 27 question
Processed row 28 question
Processed row 29 question
Processed row 30 question
Processed row 31 question
Processed row 32 question
Processed row 33 question
Processed row 34 question
Processed row 35 question
Processed row 36 question
Processed row 37 question
Processed row 38 question
Processed row 39 ques